# Data

*\[\[\[model, 123\]\]\]*

In [1]:
!ls yugioh

ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl


In [ ]:
!conda install -c anaconda python=3.8

Solving environment: failed with current_repodata.json, will retry with next repodata source.
Initial quick solve with frozen env failed.  Unfreezing env and trying again.
Solving environment: failed with current_repodata.json, will retry with next repodata source.
Solving environment: failed
Initial quick solve with frozen env failed.  Unfreezing env and trying again.
Solving environment: - 

In [15]:
!pip show scikit-learn

Name: scikit-learn
Version: 1.0.2
Summary: A set of python modules for machine learning and data mining
Home-page: http://scikit-learn.org
Author: 
Author-email: 
License: new BSD
Location: /home/shevkunov/anaconda3/lib/python3.7/site-packages
Requires: joblib, numpy, scipy, threadpoolctl
Required-by: seqeval


In [1]:
import pickle
import numpy as np
import os
import tqdm
import torch
import gc

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()

from importlib import reload
from matplotlib import pyplot as plt

import matrix_approx_zeshel

matrix_approx_zeshel = reload(matrix_approx_zeshel)

!ls  | grep matrix_approx_zeshel

2023-09-07 02:09:59.388403: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-09-07 02:09:59.418717: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-09-07 02:09:59.538325: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-09-07 02:09:59.539100: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-09-07 02:10:00.181047: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Co

matrix_approx_zeshel.py


# Open Data loader & context

In [2]:
def load_ment_to_ent_scores(directory = "yugioh", shuffle_rows = 0, full = True):
    data = list()

    for file in os.listdir(directory):
        path = f"{directory}/{file}"
        print(f"Loading file {path}")
        with open(path, "rb") as f:
            data.append(
                pickle.load(f)
            )
    data = sorted(data, key = lambda x: x["arg_dict"]["n_ment_start"])

    for i in range(len(data) - 1):
        if full:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] == data[i + 1]["arg_dict"]["n_ment_start"]
        else:
            assert data[i]["arg_dict"]["n_ment_start"] + data[i]["arg_dict"]["n_ment"] <= data[i + 1]["arg_dict"]["n_ment_start"]
        
    ment_to_ent_scores = list(map(lambda x: x["ment_to_ent_scores"], data))
    ment_to_ent_scores = np.vstack(ment_to_ent_scores)
    print("Loaded shape = ", ment_to_ent_scores.shape)
    
    if shuffle_rows:
        print(f"Shuffling... (seed = {shuffle_rows})")
        np.random.seed(shuffle_rows)
        np.random.shuffle(ment_to_ent_scores)
    
    return ment_to_ent_scores

In [3]:
class EvalContextRelevs:
    def __init__(self, relevs, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
        self.relevs = np.array(relevs)
        self.reqs_count = self.relevs.shape[0]
        self.games_count = self.relevs.shape[1]
        
        self.key_size = key_size
        self.key_games = (
            np.random.choice(np.arange(self.games_count), key_size, replace=False)
            if key_games is None else
            key_games
        )
        np.random.seed(seed)
        #np.random.shuffle(self.requests)

        self.try_det_attempts(det_attempts)
        self.train_split = int(self.reqs_count * train_size)

        assert key_size + 1 < self.train_split


        self.key_relevs = self.relevs[:key_size]
        self.train_relevs = self.relevs[key_size + 1: self.train_split]
        self.test_relevs = self.relevs[self.train_split:]

        self.slices = ["key", "train", "test"]
        print(len(self.key_relevs), len(self.train_relevs), len(self.test_relevs))

    def get_top_games(self):
        return self.relevs.mean(axis=0).argsort()[:100]

    def set_top_games_as_key(self):
        self.key_games = self.get_top_games()
        return self

    def try_det_attempts(self, det_attempts):
        def get_det(r):
            kr = r[:self.key_size, self.key_games] - r.mean()
            return np.abs(np.linalg.det(kr))

        best_i_array = np.arange(len(self.relevs))

        for i in range(det_attempts):

            r_i_array = np.arange(len(self.relevs))
            np.random.shuffle(r_i_array)

            n, o = get_det(self.relevs[r_i_array, :]), get_det(self.relevs[best_i_array, :])
            
            # print(f"try update key_reqs ({o} vs {n}...")
            if n > o:
                best_i_array = r_i_array
                print(f"updated det ({i}, {o} -> {n})")

        print("Best det = ", get_det(self.relevs[best_i_array, :]))

        self.relevs = self.relevs[best_i_array, :]
        print("Current de = ", get_det(self.relevs))
        
    def get_relevs(self, t = "train"):
        if t == "train":
            return self.train_relevs
        elif t == "key":
            return self.key_relevs
        elif t == "test":
            return self.test_relevs
        else:
            assert False
            
    def get_requests(self, t = "train"):
        if t == "train":
            return self.train_reqs
        elif t == "key":
            return self.key_reqs
        elif t == "test":
            return self.test_reqs
        else:
            assert False

In [4]:
# ctx = EvalContextRelevs(load_ment_to_ent_scores("yugioh", shuffle_rows=42), det_attempts=100)

# Games Data loader & context

In [5]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc

import matplotlib.pyplot as plt


def load(limit, raw_path = "stand/log.local.logtime2.txt", path = "log.local.logtime2.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            self.relevs = dict()
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_relevs(self, t = "train"):
            if t not in self.relevs:
                self.relevs[t] = np.array([
                    np.array([g_i[1] for g_i in r[2]])
                    for r in self.get_requests(t)
                ])
                
            return self.relevs[t]

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

In [6]:
# ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA).set_top_games_as_key()

# Models

In [7]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [8]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "matrix_l2": 0,
    "dssm_l2": 0,
    "train_popbias": False,
    "train_bias": False,
    "train_vbias": False,
    "verbose": True,
    "train_matrix": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
    "activation": "relu",
    "score_verbose": 0,
    "trainable_items": False,
    "use_keys_in_train": False
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(fit_kwargs)
        self.fit_kwargs = p
        
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[i] for i in ctx.key_games])
                    for r_i in ctx.get_relevs(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[g_i] for r in ctx.get_relevs("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        
        if self.fit_kwargs["trainable_items"]:
            self.trainable_games = tf.Variable(
                tf.convert_to_tensor(self.embed_games, dtype=tf.float32),
                trainable=True
            )
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = ctx.get_relevs("key")
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        # return self.embed_users[t]
    
    def get_game_embs(self):
        if not self.fit_kwargs["trainable_items"]:
            return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
        else:
            return self.trainable_games
            
        #return self.embed_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.p = p
        self.train_bias = p["train_bias"]
        self.train_vbias = p["train_vbias"]
        self.train_popbias = p["train_popbias"]
        self.train_matrix = p["train_matrix"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        if p["use_keys_in_train"]:
            train_user_scores = np.vstack([
                self.get_user_scores("key"),
                self.get_user_scores("train")
            ])
            train_user_embs = np.vstack([
                self.get_user_embs("key"),
                self.get_user_embs("train")
            ])
        else:
            train_user_scores = self.get_user_scores("train")
            train_user_embs = self.get_user_embs("train")

        game_embs = self.get_game_embs()
        
        initializer = tf.keras.initializers.GlorotUniform()
        values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        self.W = tf.Variable(values / 100., trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation=p["activation"]),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        if self.train_vbias:
            self.vb = tf.Variable(
                np.zeros_like(self.game_avg_scores["train"], dtype=np.float32),
                trainable = True
            ) 
        
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = (
                            game_embs[game_slice]
                            if isinstance(game_embs, np.ndarray) else
                            tf.gather(game_embs, game_slice)
                        )
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = 0
                    
                    if self.train_matrix:
                        logits += train_user_embs_ @ self.W @ game_embs_.T

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_vbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.vb
                        else:
                            logits += tf.gather(self.vb, game_slice)
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores(
                        loss_batch = (1e9 if "loss_batch" not in p else p["loss_batch"]))
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_signed":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    sf = tf.nn.softmax(logits, axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        sf,
                        -sf
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_sum(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                elif self.loss == "KL100":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    mask = np.argsort(-scores, axis=1) < 100
                    v = tf.keras.losses.KLDivergence()((scores.T >= qs.T).T, logits)
                else:
                    assert False
                    
                if self.p["c"]:
                    v += tf.reduce_mean(self.W * self.W) * p["c"]
                    
                if self.p["dssm_l2"]:
                    for weights_ in [self.u_dssm.weights, self.g_dssm.weights]:
                        for weight_ in weights_:
                            v += tf.reduce_sum(weight_ * weight_) * self.p["dssm_l2"]
                
                if self.p["matrix_l2"]:
                    v += tf.reduce_sum(self.W * self.W) * self.p["matrix_l2"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            if self.train_matrix:
                weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_vbias:
                weights += [self.vb]   
                
            if self.train_bias:
                weights += [self.b]   
                
            
            if self.p["trainable_items"]:
                weights += [self.trainable_games]
                
                
            opt.minimize(loss, weights)
            
            if p["score_verbose"] and (i % p["score_verbose"] == 0):
                print(f"\n=== Iteration {i} ===")
                for sl in self.ctx.slices:
                    print(f"slice = {sl}, score = {self.get_score(sl)}")
        print("last loss = ", loss())

    def recommend(self, t):
        logits = 0
                    
        if self.train_matrix:
            logits += self.get_user_embs(t) @ self.W @ tf.transpose(self.get_game_embs())

        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_vbias:
            logits += self.vb
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [9]:
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

In [10]:
class AnnCUR(Popular):
    def __init__(self, ctx, oracle=False, key_games=None, name=None):
        super().__init__(ctx)
        self.oracle = oracle
        self.ctx = ctx
        self.name = name

        self.key_cols_idx = np.array(sorted(ctx.key_games if key_games is None else key_games))
        rows_idx = np.arange(ctx.get_relevs("train").shape[0])
        
        self.cur = matrix_approx_zeshel.CURApprox(
            rows=torch.from_numpy(ctx.get_relevs("train")),
            cols=torch.from_numpy(ctx.get_relevs("train")[:, self.key_cols_idx]),
            row_idxs=rows_idx,
            col_idxs=self.key_cols_idx,
            approx_preference="rows",
            A=(torch.from_numpy(ctx.get_relevs("train")) if oracle else None)
        )        
        
    def __repr__(self):
        if self.name is None:
            return f"AnnCur({id(self)})"
        else:
            return f"AnnCur({self.name} - {id(self)})"
        
    def get_user_scores(self, t):
        return self.ctx.get_relevs(t)
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        key_relevs = torch.from_numpy(self.ctx.get_relevs(t)[:, self.key_cols_idx])
        return self.cur.get_complete_row(key_relevs)
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

# Evals

### open

In [11]:
import numpy as np
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import euclidean_distances

def from_labels(X, labels):
    K = list()
    for i in range(100):
        sl_i = np.arange(X.shape[0])[labels == i]
        sl = X[labels == i]
        
        if len(sl_i) == 0:
            # bug-mode
            while True:
                chosen = np.random.choice(np.arange(X.shape[0])[labels < i], size=1)[0]
                if chosen not in K:
                    K.append(chosen)
                    break
            continue
            
        center = sl.mean(axis=0).reshape(1, -1)
        best = euclidean_distances(sl, center).argmin()
        K.append(sl_i[best])
        assert labels[K[-1]] == i
    return K

def K_by_min(X):
    center = X.mean(axis=0)
    K = [euclidean_distances(X, center.reshape(1, -1)).argmax()]

    while len(K) < 100:
        K.append(euclidean_distances(X, X[K]).min(axis=1).argmax())
    return K


def eval_clustering(ctx, all_from_labels=False):
    X = ctx.get_relevs("train").T
    from sklearn.cluster import KMeans
    
    k_func = (
        (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
        if not all_from_labels else
        (lambda C : from_labels(X, C.labels_))
    )
    
    K_KMeans = k_func(
        KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
    )


    from sklearn.cluster import BisectingKMeans
    K_BisectingKMeans = k_func(
        BisectingKMeans(n_clusters=100, random_state=0).fit(X)
    )


    from sklearn.cluster import MiniBatchKMeans
    K_MiniBatchKMeans = from_labels(
        X,
        MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
    )

    from sklearn.cluster import AgglomerativeClustering
    K_AgglomerativeClustering = from_labels(
        X,
        AgglomerativeClustering(n_clusters=100).fit(X).labels_
    )

    from sklearn.cluster import SpectralCoclustering
    K_SpectralCoclustering = from_labels(
        X,
        SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralBiclustering
    K_SpectralBiclustering = from_labels(
        X,
        SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
    )

    from sklearn.cluster import SpectralClustering
    K_SpectralClusteringNN = from_labels(
        X,
        SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
    )

    K_ByMin = K_by_min(X)

    ev([
        Popular(ctx),
        AnnCUR(ctx),
        AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
        AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
        AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
        AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans"),
        AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
        AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
        AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
        AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
        AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
        AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
    ])

In [12]:
ctx = EvalContextRelevs(load_ment_to_ent_scores("yugioh", shuffle_rows=42), det_attempts=100)
ctx.key_games

Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013


array([1061, 6480, 8393, 4659, 7383, 5202, 8819, 4590, 6330, 3198, 4432,
       3290, 1684, 9299, 8683, 9847, 3941, 9056, 7010, 9880, 6488, 5896,
        638,   99, 8613, 8040, 6176, 8838, 2308,  353, 3831, 4404, 6205,
       2483, 2144, 7507, 4919, 1246, 4315, 9741, 8437, 9480, 3147, 2704,
       7958, 2348, 5238, 4963, 5035, 3963, 7641, 8425, 8565, 2420, 3684,
       9794, 4154, 3564, 3356, 6549, 2723, 2049, 2470, 4743, 2753, 1320,
       7556, 8173, 7667, 3680, 7515, 9031, 7043,  102, 1080,  993, 4531,
       6664, 5390, 1626, 9790, 9348, 6613, 1197, 3335,  518, 3231, 4165,
        181, 6659, 5078, 8483, 7535, 7019, 7330,  592, 2444, 6956, 7854,
       1368])

In [124]:
eval_clustering(ctx)

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7f31f96a8940>
np.mean(results), mse, len(results) =  0.0891637168141593 0.66850936 2260
np.mean(results), mse, len(results) =  0.09171767028627838 0.6754948 1013
0.0891637168141593 0.09171767028627838



model =  AnnCur(139852633767024)
np.mean(results), mse, len(results) =  0.4881150442477877 0.18208094 2260
np.mean(results), mse, len(results) =  0.4666929911154985 0.20506908 1013
0.4881150442477877 0.4666929911154985



model =  AnnCur(key_games=np.arange(100) - 139852612320608)
np.mean(results), mse, len(results) =  0.5052300884955753 0.17257068 2260
np.mean(results), mse, len(results) =  0.4845113524185588 0.1982153 1013
0.5052300884955753 0.4845113524185588



model =  AnnCur(random - 139852612321184)
np.mean(results), mse, len(results) =  0.492212389380531 0.17031026 2260
np.mean(results), mse, len(results) =  0.4724383020730503 0.193291 1013
0.492212389380531 0.4724383020730503



model =  AnnCur(K_KMeans - 139852612320224)
np.mean(resul

In [19]:
DATASETS = ["yugioh", "pro_wrestling", "star_trek", "doctor_who", "military"]

In [22]:
dataset = DATASETS[1]
ctx = EvalContextRelevs(load_ment_to_ent_scores(dataset, shuffle_rows=42, full=(dataset != "military")), det_attempts=100)
eval_clustering(ctx, all_from_labels=True)

Loading file pro_wrestling/ment_to_ent_scores_n_m_392_n_e_10133_all_layers_False1000.pkl
Loading file pro_wrestling/ment_to_ent_scores_n_m_500_n_e_10133_all_layers_False0.pkl
Loading file pro_wrestling/ment_to_ent_scores_n_m_500_n_e_10133_all_layers_False500.pkl
Loaded shape =  (1392, 10133)
Shuffling... (seed = 42)
updated det (1, 5.124810435461604e-33 -> 1.0217253487036523e-31)
updated det (2, 1.0217253487036523e-31 -> 1.2030961336739456e-31)
updated det (11, 1.2030961336739456e-31 -> 7.606045638566237e-29)
updated det (60, 7.606045638566237e-29 -> 3.735096148850118e-27)
Best det =  3.735096e-27
Current de =  3.735096e-27
100 873 418


/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/manifold/_spectral_embedding.py:274: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(





model =  <__main__.Popular object at 0x7f6e7d443a30>
np.mean(results), mse, len(results) =  0.23648339060710197 0.10498067 873
np.mean(results), mse, len(results) =  0.2410287081339713 0.09824814 418
0.23648339060710197 0.2410287081339713



model =  AnnCur(140114691080096)
np.mean(results), mse, len(results) =  0.48865979381443303 0.05764286 873
np.mean(results), mse, len(results) =  0.4155741626794259 0.0843202 418
0.48865979381443303 0.4155741626794259



model =  AnnCur(key_games=np.arange(100) - 140112125143424)
np.mean(results), mse, len(results) =  0.4815922107674685 0.054719765 873
np.mean(results), mse, len(results) =  0.4181818181818182 0.09742558 418
0.4815922107674685 0.4181818181818182



model =  AnnCur(random - 140112125137760)
np.mean(results), mse, len(results) =  0.49524627720504005 0.054602638 873
np.mean(results), mse, len(results) =  0.4279904306220096 0.0807769 418
0.49524627720504005 0.4279904306220096



model =  AnnCur(K_KMeans - 140112125139776)
np.mean(res

In [29]:
dataset = DATASETS[2]
ctx = EvalContextRelevs(load_ment_to_ent_scores(dataset, shuffle_rows=42, full=(dataset != "military")), det_attempts=100)
eval_clustering(ctx, all_from_labels=True)

Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False3800.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2600.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False1600.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2800.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False600.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_27_n_e_34430_all_layers_False4200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False4000.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2400.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False1200.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_layers_False2000.pkl
Loading file star_trek/ment_to_ent_scores_n_m_200_n_e_34430_all_lay

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7f26bbf7ea10>
np.mean(results), mse, len(results) =  0.08481974098704936 0.06072732 2857
np.mean(results), mse, len(results) =  0.08842395587076439 0.061203424 1269
0.08481974098704936 0.08842395587076439



model =  AnnCur(139804339061808)
np.mean(results), mse, len(results) =  0.2624851242562128 0.03921239 2857
np.mean(results), mse, len(results) =  0.23508274231678492 0.04461597 1269
0.2624851242562128 0.23508274231678492



model =  AnnCur(key_games=np.arange(100) - 139804339073232)
np.mean(results), mse, len(results) =  0.27053552677633885 0.038987625 2857
np.mean(results), mse, len(results) =  0.2488888888888889 0.044506755 1269
0.27053552677633885 0.2488888888888889



model =  AnnCur(random - 139804339072656)
np.mean(results), mse, len(results) =  0.2557297864893245 0.04027266 2857
np.mean(results), mse, len(results) =  0.22873128447596533 0.0459804 1269
0.2557297864893245 0.22873128447596533



model =  AnnCur(K_KMeans - 13980433907428

In [28]:
dataset = DATASETS[3]
ctx = EvalContextRelevs(load_ment_to_ent_scores(dataset, shuffle_rows=42, full=(dataset != "military")), det_attempts=100)
eval_clustering(ctx, all_from_labels=True)

Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False2600.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False1800.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False0600.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False2000.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False2200.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False1200.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False3000.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False3200.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False3400.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False1600.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e_40281_all_layers_False0000.pkl
Loading file doctor_who/ment_to_ent_scores_n_m_200_n_e

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7f26bbf7f2e0>
np.mean(results), mse, len(results) =  0.08161541311596889 0.046746448 2699
np.mean(results), mse, len(results) =  0.08209166666666667 0.04624705 1200
0.08161541311596889 0.08209166666666667



model =  AnnCur(139804339071024)
np.mean(results), mse, len(results) =  0.22464246017043346 0.034886137 2699
np.mean(results), mse, len(results) =  0.19922499999999999 0.03896426 1200
0.22464246017043346 0.19922499999999999



model =  AnnCur(key_games=np.arange(100) - 139804339070736)
np.mean(results), mse, len(results) =  0.2295331604297888 0.034667376 2699
np.mean(results), mse, len(results) =  0.197475 0.039356913 1200
0.2295331604297888 0.197475



model =  AnnCur(random - 139804339061184)
np.mean(results), mse, len(results) =  0.21482400889218228 0.035653345 2699
np.mean(results), mse, len(results) =  0.19191666666666668 0.040786836 1200
0.21482400889218228 0.19191666666666668



model =  AnnCur(K_KMeans - 139804339071888)
np.mean(res

In [23]:
dataset = DATASETS[4]
ctx = EvalContextRelevs(load_ment_to_ent_scores(dataset, shuffle_rows=42, full=(dataset != "military")), det_attempts=100)
# eval_clustering(ctx, all_from_labels=True)

Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1400.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0900.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0100.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1700.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1900.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_l

In [15]:
ev([
    Popular(ctx),
    AnnCUR(ctx),
    AnnCUR(ctx, key_games=np.arange(100), name="key_games=np.arange(100)"),
])




model =  <__main__.Popular object at 0x7fe066bcd390>
np.mean(results), mse, len(results) =  0.1148385053831539 0.01570362 1579
np.mean(results), mse, len(results) =  0.11273611111111109 0.015634432 720
0.1148385053831539 0.11273611111111109



model =  AnnCur(140601773058048)
np.mean(results), mse, len(results) =  0.30521849271690943 0.008218333 1579
np.mean(results), mse, len(results) =  0.24825000000000003 0.010689547 720
0.30521849271690943 0.24825000000000003



model =  AnnCur(key_games=np.arange(100) - 140601773060928)
np.mean(results), mse, len(results) =  0.31364787840405317 0.008147016 1579
np.mean(results), mse, len(results) =  0.2599166666666667 0.011355296 720
0.31364787840405317 0.2599166666666667


In [17]:
gc.collect()

0

In [15]:
X = ctx.get_relevs("train").T
all_from_labels=True

from sklearn.cluster import KMeans
    
k_func = (
    (lambda C : euclidean_distances(X, C.cluster_centers_).argmin(axis=0))
    if not all_from_labels else
    (lambda C : from_labels(X, C.labels_))
)

In [19]:
K_KMeans = k_func(
    KMeans(n_clusters=100, random_state=0).fit(X)  #, n_init="auto").fit(X)
)

ev([
    AnnCUR(ctx, key_games=np.random.choice(np.arange(X.shape[0]), size=100, replace=False), name="random"),
    AnnCUR(ctx, key_games=K_KMeans, name="K_KMeans"),
])

del K_KMeans
gc.collect()

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  AnnCur(random - 140601771535248)
np.mean(results), mse, len(results) =  0.29692210259658014 0.008513262 1579
np.mean(results), mse, len(results) =  0.2454861111111111 0.013692013 720
0.29692210259658014 0.2454861111111111



model =  AnnCur(K_KMeans - 140601772938176)
np.mean(results), mse, len(results) =  0.36362887903736546 0.0071787625 1579
np.mean(results), mse, len(results) =  0.30419444444444443 0.011677039 720
0.36362887903736546 0.30419444444444443


9

In [20]:
from sklearn.cluster import BisectingKMeans
K_BisectingKMeans = k_func(
    BisectingKMeans(n_clusters=100, random_state=0).fit(X)
)

ev([
    AnnCUR(ctx, key_games=K_BisectingKMeans, name="K_BisectingKMeans")
])

del K_BisectingKMeans
gc.collect()




model =  AnnCur(K_BisectingKMeans - 140601772457552)
np.mean(results), mse, len(results) =  0.329537682077264 0.0078093973 1579
np.mean(results), mse, len(results) =  0.27516666666666667 0.010857157 720
0.329537682077264 0.27516666666666667


0

In [21]:
from sklearn.cluster import MiniBatchKMeans
K_MiniBatchKMeans = from_labels(
    X,
    MiniBatchKMeans(n_clusters=100, random_state=0).fit(X).labels_
)

ev([
    AnnCUR(ctx, key_games=K_MiniBatchKMeans, name="K_MiniBatchKMeans"),
])

del K_MiniBatchKMeans
gc.collect()

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  AnnCur(K_MiniBatchKMeans - 140601771533760)
np.mean(results), mse, len(results) =  0.34352121595946805 0.0074226977 1579
np.mean(results), mse, len(results) =  0.2826388888888889 0.012286168 720
0.34352121595946805 0.2826388888888889


0

In [29]:
from sklearn.cluster import AgglomerativeClustering

In [31]:
from sklearn.cluster import AgglomerativeClustering
K_AgglomerativeClustering = from_labels(
    X,
    AgglomerativeClustering(n_clusters=100,
                           compute_full_tree=False).fit(X).labels_
)

ev([
    AnnCUR(ctx, key_games=K_AgglomerativeClustering, name="K_AgglomerativeClustering"),
])

del K_AgglomerativeClustering
gc.collect()




model =  AnnCur(K_AgglomerativeClustering - 140112527127744)
np.mean(results), mse, len(results) =  0.2984547181760608 0.00875553 1579
np.mean(results), mse, len(results) =  0.24483333333333335 0.012242817 720
0.2984547181760608 0.24483333333333335


0

In [28]:
ctx.relevs.shape

(2400, 104520)

In [27]:
from sklearn.cluster import SpectralCoclustering
K_SpectralCoclustering = from_labels(
    X,
    SpectralCoclustering(n_clusters=100, random_state=0).fit(X).row_labels_
)

ev([
    AnnCUR(ctx, key_games=K_SpectralCoclustering, name="K_SpectralCoclustering"),
])

del K_SpectralCoclustering
gc.collect()




model =  AnnCur(K_SpectralCoclustering - 140112524904240)
np.mean(results), mse, len(results) =  0.31143128562381256 0.008089031 1579
np.mean(results), mse, len(results) =  0.25506944444444446 0.010661272 720
0.31143128562381256 0.25506944444444446


0

In [26]:
from sklearn.cluster import SpectralBiclustering
K_SpectralBiclustering = from_labels(
    X,
    SpectralBiclustering(n_clusters=100, random_state=0).fit(X).row_labels_
)

ev([
    AnnCUR(ctx, key_games=K_SpectralBiclustering, name="K_SpectralBiclustering"),
])

del K_SpectralBiclustering
gc.collect()




model =  AnnCur(K_SpectralBiclustering - 140114696495424)
np.mean(results), mse, len(results) =  0.3090880303989867 0.008293898 1579
np.mean(results), mse, len(results) =  0.2532916666666667 0.0112146 720
0.3090880303989867 0.2532916666666667


0

In [25]:
from sklearn.cluster import SpectralClustering
K_SpectralClusteringNN = from_labels(
    X,
    SpectralClustering(n_clusters=100, random_state=0, affinity="nearest_neighbors", n_jobs=-1).fit(X).labels_
)


ev([
    AnnCUR(ctx, key_games=K_SpectralClusteringNN, name="K_SpectralClusteringNN"),
])

del K_SpectralClusteringNN
gc.collect()




model =  AnnCur(K_SpectralClusteringNN - 140112527134720)
np.mean(results), mse, len(results) =  0.3124889170360988 0.008279293 1579
np.mean(results), mse, len(results) =  0.2507361111111111 0.013905897 720
0.3124889170360988 0.2507361111111111


0

In [24]:
K_ByMin = K_by_min(X)

ev([
    AnnCUR(ctx, key_games=K_ByMin, name="K_ByMin"),
])

del K_ByMin
gc.collect()




model =  AnnCur(K_ByMin - 140112125144048)
np.mean(results), mse, len(results) =  0.3059024699176694 0.008322237 1579
np.mean(results), mse, len(results) =  0.24830555555555556 0.011385426 720
0.3059024699176694 0.24830555555555556


1139

In [20]:
dataset = DATASETS[4]
ctx = EvalContextRelevs(
    load_ment_to_ent_scores(dataset, shuffle_rows=42, full=(dataset != "military")), det_attempts=100
).set_top_games_as_key()
# eval_clustering(ctx, all_from_labels=True)

Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1400.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0900.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0100.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1700.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1500.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False1900.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False2300.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_layers_False0200.pkl
Loading file military/ment_to_ent_scores_n_m_100_n_e_104520_all_l

In [21]:
ev([
    AnnCUR(ctx)
])




model =  AnnCur(139869535055984)
np.mean(results), mse, len(results) =  0.24728942368587714 0.0100673335 1579
np.mean(results), mse, len(results) =  0.19073611111111113 0.016318358 720
0.24728942368587714 0.19073611111111113


In [33]:
for dataset in DATASETS:
    ctx = EvalContextRelevs(
        load_ment_to_ent_scores(dataset, shuffle_rows=42, full=(dataset != "military")), det_attempts=100
    ).set_top_games_as_key()
    ev([
        AnnCUR(ctx)
    ])

Loading file yugioh/ment_to_ent_scores_n_m_374_n_e_10031_all_layers_False3000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False2000.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False500.pkl
Loading file yugioh/ment_to_ent_scores_n_m_500_n_e_10031_all_layers_False1500.pkl
Loaded shape =  (3374, 10031)
Shuffling... (seed = 42)
updated det (4, 6.137885635798218e+30 -> 2.524378507319208e+32)
updated det (8, 2.524378507319208e+32 -> 5.48097506735188e+34)
updated det (11, 5.48097506735188e+34 -> 4.6749692824069873e+36)
Best det =  4.6749693e+36
Current de =  4.6749693e+36
100 2260 1013



model =  AnnCur(139804338235104)
np.mean(results), mse, len(results) =  0.2658451327433628 0.44091052 2260
np.mea

FileNotFoundError: [Errno 2] No such file or directory: 'military'

### games

In [12]:
L = 7000
N = 1000
DA = 50

In [33]:
!ls -lah

total 3.9G
drwxr-xr-x 10 shevkunov shevkunov 4.0K Sep  6 11:48 .
drwxr-xr-x 56 shevkunov shevkunov 4.0K May  6 01:49 ..
drwxrwxr-x  2 shevkunov shevkunov 4.0K Aug 20 15:53 doctor_who
-rwxr-xr-x  1 shevkunov shevkunov 108M Apr 14 15:02 games-all.json
drwxr-xr-x  2 shevkunov shevkunov 4.0K Aug 24 19:45 .ipynb_checkpoints
-rw-rw-r--  1 shevkunov shevkunov 1.3G May 19 12:26 log.local.bin
-rwxr-xr-x  1 shevkunov shevkunov 691M May 19 11:32 log.local.bin-old
-rw-rw-r--  1 shevkunov shevkunov 1.8G Jul 22 00:21 log.local.logtime2.bin
-rw-r--r--  1 shevkunov shevkunov 5.5K Jul  9 17:54 matrix_approx_zeshel.py
drwxrwxr-x  2 shevkunov shevkunov 4.0K Sep  2 17:08 military
-rwxr-xr-x  1 shevkunov shevkunov 345K Feb  8  2023 proof-of-concept.ipynb
-rw-rw-r--  1 shevkunov shevkunov 438K Jul 12 12:23 proof-of-concept-open-data-round1.ipynb
-rw-rw-r--  1 shevkunov shevkunov 898K Jul 30 16:51 proof-of-concept-open-data-round2-anncurcomparison.ipynb
-rw-rw-r--  1 shevkunov shevkunov  60K Aug 24 13:30 pro

In [13]:
ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA) # .set_top_games_as_key()
ev([
    AnnCUR(ctx)
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.628053728316076e-117
2.628053728316076e-117
101 4644 2034



model =  AnnCur(139929851077648)
np.mean(results), mse, len(results) =  0.6764900947459087 1.9841502796717772 4644
np.mean(results), mse, len(results) =  0.6652409046214356 2.8454261374203558 2034
0.6764900947459087 0.6652409046214356


In [16]:
ctx = load(L, raw_path = "stand/log.local.txt", seed=17, det_attempts=DA).set_top_games_as_key()

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.628053728316076e-117
2.628053728316076e-117
101 4644 2034


In [27]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7fd33860bb20>
np.mean(results), mse, len(results) =  0.5083527131782946 2.1090857474029794 4644
np.mean(results), mse, len(results) =  0.5076548672566372 2.214983388749716 2034
0.5083527131782946 0.5076548672566372



model =  AnnCur(140545160627632)
np.mean(results), mse, len(results) =  0.7693260120585702 0.5653017362404418 4644
np.mean(results), mse, len(results) =  0.762251720747296 0.5927785494345469 2034
0.7693260120585702 0.762251720747296



model =  AnnCur(key_games=np.arange(100) - 140531535266816)
np.mean(results), mse, len(results) =  0.6579565030146425 1.9767935375740953 4644
np.mean(results), mse, len(results) =  0.6489528023598821 6.906013812566882 2034
0.6579565030146425 0.6489528023598821



model =  AnnCur(random - 140531535278288)
np.mean(results), mse, len(results) =  0.6810099052540913 1.9702658686057843 4644
np.mean(results), mse, len(results) =  0.6697246804326451 2.180510915221889 2034
0.6810099052540913 0.66972468043264

In [34]:
!ls stand

log.local.logtime2.txt.old  log.local.txt  log.local.txt.tar.gz


In [13]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA)

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.609521994482665e-120
2.609521994482665e-120
101 4769 2088


In [14]:
ev([
    Popular(ctx),
    AnnCUR(ctx)
])




model =  <__main__.Popular object at 0x7ff62130f6d0>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(140695785624080)
np.mean(results), mse, len(results) =  0.5976535961417488 0.17482987219177043 4769
np.mean(results), mse, len(results) =  0.5860488505747127 0.19354859381230788 2088
0.5976535961417488 0.5860488505747127


In [15]:
ctx = load(L, raw_path = "stand/log.local.logtime2.txt",
           path="log.local.logtime2.true.bin", seed=17, det_attempts=DA).set_top_games_as_key()

/tmp/ipykernel_690140/664571995.py:33: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for i, line in tqdm.tqdm_notebook(enumerate(f)):


0it [00:00, ?it/s]

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.609521994482665e-120
2.609521994482665e-120
101 4769 2088


In [16]:
ev([
    Popular(ctx),
    AnnCUR(ctx)
])




model =  <__main__.Popular object at 0x7f3894f410c0>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(139872009046960)
np.mean(results), mse, len(results) =  0.6778381211994128 0.24971886538105065 4769
np.mean(results), mse, len(results) =  0.669477969348659 0.27141239614050905 2088
0.6778381211994128 0.669477969348659


In [17]:
eval_clustering(ctx, all_from_labels=True)

/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(
/home/shevkunov/anaconda3/lib/python3.10/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 3 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(





model =  <__main__.Popular object at 0x7f3677c5d450>
np.mean(results), mse, len(results) =  0.2893499685468652 5.200994032516979 4769
np.mean(results), mse, len(results) =  0.2885536398467433 5.111057961798107 2088
0.2893499685468652 0.2885536398467433



model =  AnnCur(139880993919168)
np.mean(results), mse, len(results) =  0.6778381211994128 0.24971886538105065 4769
np.mean(results), mse, len(results) =  0.669477969348659 0.27141239614050905 2088
0.6778381211994128 0.669477969348659



model =  AnnCur(key_games=np.arange(100) - 139886535450944)
np.mean(results), mse, len(results) =  0.568666387083246 0.2101395930999089 4769
np.mean(results), mse, len(results) =  0.5609003831417625 0.2306177104605779 2088
0.568666387083246 0.5609003831417625



model =  AnnCur(random - 139886535448400)
np.mean(results), mse, len(results) =  0.5932228978821555 0.1733168362865605 4769
np.mean(results), mse, len(results) =  0.584176245210728 0.1904597187905242 2088
0.5932228978821555 0.58417624521072